## SQL Analysis

All queries below are run using SQLite in Python via the `sqlite3` library. The complaints data is loaded from the cleaned CSV into an in-memory SQLite database, along with three supplementary lookup tables: `borough_lookup` (borough codes to names), `disposition_lookup` (disposition codes to categories), and `borough_income` (US Census ACS 2023 median income and poverty rate by borough).

In [7]:
import sqlite3
import pandas as pd

complaints = pd.read_csv("/home/hkarlin/complaints_clean.csv")
complaints["date_entered"] = pd.to_datetime(complaints["date_entered"])
complaints["disposition_date"] = pd.to_datetime(complaints["disposition_date"], errors="coerce")

conn = sqlite3.connect(":memory:")
complaints.to_sql("complaints", conn, if_exists="replace", index=False)

119491

In [8]:
borough_lookup = pd.DataFrame({
    "borough_code": ["1", "2", "3", "4", "5"],
    "borough_name": ["Manhattan", "Bronx", "Brooklyn", "Queens", "Staten Island"]
})
borough_lookup.to_sql("borough_lookup", conn, if_exists="replace", index=False)


disposition_lookup = (
    complaints[["disposition_code", "disposition_category"]]
    .dropna()
    .drop_duplicates()
)
disposition_lookup.to_sql("disposition_lookup", conn, if_exists="replace", index=False)


borough_income = pd.DataFrame({
    "borough": ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"],
    "median_household_income": [93000, 63000, 70000, 43000, 80000],
    "poverty_rate_pct": [12.8, 19.2, 12.9, 29.0, 10.5]
})
borough_income.to_sql("borough_income", conn, if_exists="replace", index=False)

print("Setup complete")
pd.read_sql("SELECT COUNT(*) as total_rows FROM complaints", conn)

Setup complete


,total_rows
0,119491


### Query 1: Complaint Volume and Enforcement Outcomes by Borough
**Type:** GROUP BY

This query establishes the borough-level baseline — how many complaints each borough receives, how long they take to resolve on average, and what share result in access denial. Queens emerges as the most challenging borough to enforce in, with the highest access denial rate (30.9%) and the slowest average resolution time (71.3 days).

In [9]:
pd.read_sql("""
    SELECT 
        borough,
        COUNT(*) AS total_complaints,
        ROUND(AVG(days_to_resolve), 1) AS avg_days_to_resolve,
        SUM(CASE WHEN disposition_category = 'Access Denied' THEN 1 ELSE 0 END) AS access_denied_count,
        ROUND(100.0 * SUM(CASE WHEN disposition_category = 'Access Denied' THEN 1 ELSE 0 END) / COUNT(*), 1) AS access_denied_pct
    FROM complaints
    WHERE borough != 'Unknown'
    GROUP BY borough
    ORDER BY total_complaints DESC
""", conn)

,borough,total_complaints,avg_days_to_resolve,access_denied_count,access_denied_pct
0,Brooklyn,39908,46.8,7445,18.7
1,Queens,33024,71.3,10207,30.9
2,Manhattan,24084,26.4,2123,8.8
3,Bronx,16805,45.1,2222,13.2
4,Staten Island,5670,61.9,1069,18.9


### Query 2: Top 15 Complaint Categories by Volume and Resolution Time
**Type:** GROUP BY

This query identifies the most common complaint types and how long each takes to close. Illegal Conversion dominates on both dimensions — 13,309 complaints and an average of 135.6 days to close — making it the single most operationally significant complaint type for the DOB.

In [10]:
pd.read_sql("""
    SELECT 
        complaint_category_desc,
        COUNT(*) AS complaint_count,
        ROUND(AVG(days_to_resolve), 1) AS avg_days_to_close,
        ROUND(MIN(days_to_resolve), 0) AS min_days,
        ROUND(MAX(days_to_resolve), 0) AS max_days
    FROM complaints
    WHERE complaint_category_desc IS NOT NULL
    GROUP BY complaint_category_desc
    ORDER BY complaint_count DESC
    LIMIT 15
""", conn)

,complaint_category_desc,complaint_count,avg_days_to_close,min_days,max_days
0,Illegal Conversion,13309,135.6,0.0,449.0
1,Work Without a Permit – Occupied Multiple Dwel...,6073,5.7,0.0,462.0
2,Elevator: Single Device on Property,5358,92.2,0.0,405.0
3,Boiler: Defective/Inoperative/No Permit,4605,21.6,0.0,350.0
4,Sidewalk Shed/Scaffold – Inadequate/No Permit,4414,5.9,0.0,407.0
5,Building Shaking/Vibrating/Structural Stability,4046,10.8,0.0,471.0
6,Construction Safety Compliance (CSC) Action,3836,19.4,0.0,386.0
7,Construction Enforcement Work Order,3514,27.0,0.0,502.0
8,CSE: Tracking Compliance,3467,8.1,0.0,392.0
9,Failure to Maintain,3400,30.4,0.0,497.0


### Query 3: Monthly Complaint Volume and Average Resolution Time (2025)
**Type:** GROUP BY

This query breaks down complaint filings and resolution time across all 12 months of 2025. Complaint volume peaks in October (11,367) and July (10,725), while resolution time is highest mid-year (June–July, ~59–60 days) and drops sharply in Q4. The December average of 33.8 days reflects complaints filed recently that closed quickly — longer-running cases were still open at year end.

In [11]:
pd.read_sql("""
    SELECT 
        strftime('%Y', date_entered) AS year,
        strftime('%m', date_entered) AS month,
        COUNT(*) AS complaint_count,
        ROUND(AVG(days_to_resolve), 1) AS avg_days_to_resolve
    FROM complaints
    GROUP BY year, month
    ORDER BY year, month
""", conn)

,year,month,complaint_count,avg_days_to_resolve
0,2025,01,9267,51.4
1,2025,02,8590,47.4
2,2025,03,9999,51.0
3,2025,04,9986,54.4
4,2025,05,10096,57.9
5,2025,06,10627,59.2
6,2025,07,10725,60.3
7,2025,08,10183,53.0
8,2025,09,9904,46.0
9,2025,10,11367,44.0


### Query Top Complaint Types by Borough
**Type:** JOIN

This query joins the complaints table to `borough_lookup` on the community board prefix, then groups by borough and complaint type to surface the highest-volume combinations. Filtering out null descriptions and combinations with fewer than 500 complaints keeps the results meaningful. The standout finding is that Illegal Conversion in Queens averages 163.5 days to resolve — nearly 40% slower than the same complaint type in Brooklyn (117.7 days) — reinforcing that Queens faces structural enforcement barriers beyond just complaint volume.

In [15]:
pd.read_sql("""
    SELECT 
        bl.borough_name,
        c.complaint_category_desc,
        COUNT(*) AS total_complaints,
        ROUND(AVG(c.days_to_resolve), 1) AS avg_days_to_resolve
    FROM complaints c
    JOIN borough_lookup bl ON SUBSTR(c.community_board, 1, 1) = bl.borough_code
    WHERE c.complaint_category_desc IS NOT NULL
    GROUP BY bl.borough_name, c.complaint_category_desc
    HAVING COUNT(*) > 500
    ORDER BY total_complaints DESC
    LIMIT 10
""", conn)

,borough_name,complaint_category_desc,total_complaints,avg_days_to_resolve
0,Queens,Illegal Conversion,7318,163.5
1,Brooklyn,Illegal Conversion,2980,117.7
2,Brooklyn,Work Without a Permit – Occupied Multiple Dwel...,2856,5.9
3,Manhattan,Sidewalk Shed/Scaffold – Inadequate/No Permit,1929,3.6
4,Manhattan,Work Without a Permit – Occupied Multiple Dwel...,1756,4.7
5,Bronx,Elevator: Single Device on Property,1733,115.7
6,Bronx,Illegal Conversion,1609,79.6
7,Brooklyn,Boiler: Defective/Inoperative/No Permit,1507,18.5
8,Brooklyn,Construction Safety Compliance (CSC) Action,1492,20.1
9,Brooklyn,Construction: Contrary to Approved Plans,1485,65.7


### Query 5: Complaint Outcomes by Category Using Disposition Lookup Table
**Type:** JOIN

This query joins the complaints table to the `disposition_lookup` table on `disposition_code` to retrieve the canonical disposition category for each complaint, then summarizes volume and average resolution time by outcome type. "Resolved / Compliant" is the most common outcome, but "Violation / Enforcement Action" complaints take significantly longer to close on average.

In [16]:
pd.read_sql("""
    SELECT 
        dl.disposition_category,
        COUNT(*) AS complaint_count,
        ROUND(AVG(c.days_to_resolve), 1) AS avg_days_to_resolve,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM complaints WHERE disposition_code IS NOT NULL), 1) AS pct_of_total
    FROM complaints c
    JOIN disposition_lookup dl ON c.disposition_code = dl.disposition_code
    GROUP BY dl.disposition_category
    ORDER BY complaint_count DESC
""", conn)

,disposition_category,complaint_count,avg_days_to_resolve,pct_of_total
0,Resolved / Compliant,53016,38.2,44.6
1,Violation / Enforcement Action,28594,37.5,24.1
2,Access Denied,23066,102.9,19.4
3,Closed – No Action,6247,27.7,5.3
4,Pending / Monitoring,3467,55.5,2.9
5,Referred to Other Agency,2614,18.6,2.2
6,Emergency / Vacate,1479,22.7,1.2
7,Assigned to Unit,322,44.5,0.3


### Query 6: Three-Table Join — Borough Lookup, Complaints, and Income Data
**Type:** JOIN (three tables)

This query joins all three supplementary tables together: `borough_lookup` maps the community board prefix to a borough name, which is then joined to `borough_income` for socioeconomic context. This demonstrates how the borough derivation logic can be validated and enriched entirely in SQL without relying on the pre-computed `borough` column.

In [17]:
pd.read_sql("""
    SELECT 
        bl.borough_name,
        COUNT(*) AS total_complaints,
        ROUND(AVG(c.days_to_resolve), 1) AS avg_days_to_resolve,
        bi.median_household_income,
        bi.poverty_rate_pct
    FROM complaints c
    JOIN borough_lookup bl ON SUBSTR(c.community_board, 1, 1) = bl.borough_code
    JOIN borough_income bi ON bl.borough_name = bi.borough
    GROUP BY bl.borough_name, bi.median_household_income, bi.poverty_rate_pct
    ORDER BY total_complaints DESC
""", conn)

,borough_name,total_complaints,avg_days_to_resolve,median_household_income,poverty_rate_pct
0,Brooklyn,39908,46.8,63000,19.2
1,Queens,33024,71.3,70000,12.9
2,Manhattan,24084,26.4,93000,12.8
3,Bronx,16805,45.1,43000,29.0
4,Staten Island,5670,61.9,80000,10.5


### Query 7: Borough Rankings by Resolution Speed and Access Denial Rate
**Type:** Window Function (RANK)

Using `RANK() OVER`, this query ranks each borough on two dimensions: slowest average resolution time and highest access denial rate. Queens ranks #1 on both measures, making it a clear outlier. The window function allows these rankings to be computed alongside raw values in a single pass — no self-joins required.

In [18]:
pd.read_sql("""
    SELECT
        borough,
        total_complaints,
        avg_days_to_resolve,
        access_denied_pct,
        RANK() OVER (ORDER BY avg_days_to_resolve DESC) AS rank_slowest_resolution,
        RANK() OVER (ORDER BY access_denied_pct DESC) AS rank_most_access_denied
    FROM (
        SELECT 
            borough,
            COUNT(*) AS total_complaints,
            ROUND(AVG(days_to_resolve), 1) AS avg_days_to_resolve,
            ROUND(100.0 * SUM(CASE WHEN disposition_category = 'Access Denied' THEN 1 ELSE 0 END) / COUNT(*), 1) AS access_denied_pct
        FROM complaints
        WHERE borough != 'Unknown'
        GROUP BY borough
    )
    ORDER BY rank_slowest_resolution
""", conn)

,borough,total_complaints,avg_days_to_resolve,access_denied_pct,rank_slowest_resolution,rank_most_access_denied
0,Queens,33024,71.3,30.9,1,1
1,Staten Island,5670,61.9,18.9,2,2
2,Brooklyn,39908,46.8,18.7,3,3
3,Bronx,16805,45.1,13.2,4,4
4,Manhattan,24084,26.4,8.8,5,5


### Query 8: Monthly Running Total of Complaints with Rolling Average Resolution Time
**Type:** Window Function (SUM OVER, AVG OVER)

This query uses `SUM() OVER` to compute a cumulative complaint count as months progress through 2025, and `AVG() OVER` with a 3-month rolling window to smooth the resolution time trend. The rolling average reveals a clear pattern: resolution time rises through summer and falls sharply in Q4.

In [19]:
pd.read_sql("""
    SELECT
        month,
        complaint_count,
        avg_days_to_resolve,
        SUM(complaint_count) OVER (ORDER BY month) AS running_total_complaints,
        ROUND(AVG(avg_days_to_resolve) OVER (
            ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 1) AS rolling_3mo_avg_days
    FROM (
        SELECT
            strftime('%m', date_entered) AS month,
            COUNT(*) AS complaint_count,
            ROUND(AVG(days_to_resolve), 1) AS avg_days_to_resolve
        FROM complaints
        GROUP BY month
    )
    ORDER BY month
""", conn)

,month,complaint_count,avg_days_to_resolve,running_total_complaints,rolling_3mo_avg_days
0,01,9267,51.4,9267,51.4
1,02,8590,47.4,17857,49.4
2,03,9999,51.0,27856,49.9
3,04,9986,54.4,37842,50.9
4,05,10096,57.9,47938,54.4
5,06,10627,59.2,58565,57.2
6,07,10725,60.3,69290,59.1
7,08,10183,53.0,79473,57.5
8,09,9904,46.0,89377,53.1
9,10,11367,44.0,100744,47.7


### Query 9: Complaint Categories with Above-Citywide-Average Resolution Time
**Type:** Subquery

A subquery computes the citywide average resolution time, which is then used in the outer WHERE clause to filter for complaint categories that exceed it. This surfaces the complaint types that are most likely to become long-running enforcement problems — a useful prioritization tool for DOB management.

In [20]:
pd.read_sql("""
    SELECT
        complaint_category_desc,
        COUNT(*) AS complaint_count,
        ROUND(AVG(days_to_resolve), 1) AS avg_days_to_resolve,
        ROUND(AVG(days_to_resolve) - (
            SELECT AVG(days_to_resolve) FROM complaints WHERE days_to_resolve IS NOT NULL
        ), 1) AS days_above_citywide_avg
    FROM complaints
    WHERE days_to_resolve > (
        SELECT AVG(days_to_resolve) FROM complaints WHERE days_to_resolve IS NOT NULL
    )
    AND complaint_category_desc IS NOT NULL
    GROUP BY complaint_category_desc
    ORDER BY avg_days_to_resolve DESC
    LIMIT 15
""", conn)

,complaint_category_desc,complaint_count,avg_days_to_resolve,days_above_citywide_avg
0,Sign Falling/Illegal Erection,7,268.9,218.9
1,Mechanical Demolition – Illegal,2,251.0,201.1
2,Façade – Unsafe Notification,30,228.8,178.9
3,Storefront/Business Sign – Illegal,500,199.9,150.0
4,Building Shaking/Vibrating/Structural Stability,195,193.7,143.8
5,Failure to Comply with Vacate Order,2,193.0,143.1
6,Unlicensed/Illegal Work In-Progress,65,191.5,141.6
7,Advertising Sign/Billboard – Illegal,177,189.2,139.3
8,Illegal Commercial Use in Residential Zone,521,188.8,138.9
9,Work Without a Permit – Occupied Multiple Dwel...,141,184.8,134.9


### Query 10: Boroughs with Above-Citywide-Average Access Denial Rate
**Type:** Subquery (HAVING)

This query uses a subquery inside a `HAVING` clause to filter the grouped results to only those boroughs whose access denial rate exceeds the citywide average. Queens clears the threshold by nearly double — its 30.9% rate is far above the citywide figure.

In [21]:
pd.read_sql("""
    SELECT
        borough,
        COUNT(*) AS total_complaints,
        ROUND(100.0 * SUM(CASE WHEN disposition_category = 'Access Denied' THEN 1 ELSE 0 END) / COUNT(*), 1) AS access_denied_pct
    FROM complaints
    WHERE borough != 'Unknown'
    GROUP BY borough
    HAVING access_denied_pct > (
        SELECT ROUND(100.0 * SUM(CASE WHEN disposition_category = 'Access Denied' THEN 1 ELSE 0 END) / COUNT(*), 1)
        FROM complaints
        WHERE borough != 'Unknown'
    )
""", conn)

,borough,total_complaints,access_denied_pct
0,Queens,33024,30.9


### Query 11: Violation and Enforcement Action Rate by Borough
**Type:** GROUP BY

This query calculates what percentage of complaints in each borough result in a Violation or Enforcement Action — the DOB's most active response. Rates are broadly consistent across boroughs (22–28%), but the Bronx leads slightly. Combined with the Bronx's low access denial rate, this suggests inspectors there are both gaining access and acting on what they find at higher rates than other boroughs.

In [22]:
pd.read_sql("""
    SELECT
        borough,
        COUNT(*) AS total_complaints,
        SUM(CASE WHEN disposition_category = 'Violation / Enforcement Action' THEN 1 ELSE 0 END) AS violation_count,
        ROUND(100.0 * SUM(CASE WHEN disposition_category = 'Violation / Enforcement Action' THEN 1 ELSE 0 END) / COUNT(*), 1) AS violation_pct,
        ROUND(100.0 * SUM(CASE WHEN disposition_category = 'Resolved / Compliant' THEN 1 ELSE 0 END) / COUNT(*), 1) AS resolved_pct
    FROM complaints
    WHERE borough != 'Unknown'
    GROUP BY borough
    ORDER BY violation_pct DESC
""", conn)

,borough,total_complaints,violation_count,violation_pct,resolved_pct
0,Bronx,16805,4645,27.6,44.8
1,Brooklyn,39908,9790,24.5,43.6
2,Staten Island,5670,1369,24.1,45.3
3,Manhattan,24084,5543,23.0,55.5
4,Queens,33024,7247,21.9,36.7


### Query 12: Monthly Trend in Illegal Conversion Complaints
**Type:** GROUP BY

Illegal Conversion is the most-filed complaint type in the dataset (13,309 complaints) and the slowest to resolve (135.6 days on average). This query isolates its monthly filing trend to understand whether it spikes seasonally. Volume is relatively stable month-to-month, suggesting Illegal Conversion is a persistent structural issue rather than a seasonal one — reinforcing the case for a dedicated enforcement unit rather than surge staffing.

In [38]:
pd.read_sql("""
    SELECT
        strftime('%m', date_entered) AS month,
        COUNT(*) AS complaint_count,
        ROUND(AVG(days_to_resolve), 1) AS avg_days_to_resolve,
        ROUND(100.0 * SUM(CASE WHEN disposition_category = 'Access Denied' THEN 1 ELSE 0 END) / COUNT(*), 1) AS access_denied_pct
    FROM complaints
    WHERE complaint_category_desc = 'Illegal Conversion'
    GROUP BY month
    ORDER BY month
""", conn)

,month,complaint_count,avg_days_to_resolve,access_denied_pct
0,01,947,142.3,70.1
1,02,818,152.2,70.0
2,03,937,150.6,67.6
3,04,1058,165.8,68.6
4,05,1103,161.0,68.2
5,06,1247,151.4,68.0
6,07,1466,151.6,70.3
7,08,1345,136.3,70.0
8,09,1110,120.2,70.2
9,10,1567,113.0,72.6
